In [0]:
spark.sql("CREATE schema if not exists bronze")
spark.sql("create schema if not exists silver")
spark.sql("create schema if not exists gold")

print("schemas created: bronze, silver, gold")

schemas created: bronze, silver, gold


## Step 2: Bronze Layer - Raw Ingestion

In [0]:
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, trim, upper, lower, initcap, count, sum as spark_sum, avg, row_number, to_date, date_format, monotonically_increasing_id
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StringType, IntegerType, StructField, StringType, DoubleType, DataType
)
from datetime import datetime

### Generate Simulated Banking Transactions

In [0]:
transactions_data = [
    ("TXN001", "ACC100", 250.00, "debit", "SuperMart", "2024-03-01 09:15:00", "ZAR", "pos"),
    ("TXN002", "ACC101", 1500.00, "credit", "Salary Deposit", "2024-03-01 08:00:00", "ZAR", "eft"),
    ("TXN003", "ACC100", 89.99, "debit", "Netflix", "2024-03-01 12:00:00", "ZAR", "online"),
    ("TXN004", "ACC102", 3200.00, "debit", "Rent Payment", "2024-03-01 07:00:00", "ZAR", "eft"),
    ("TXN005", "ACC101", 45.50, "debit", "Coffee Shop", "2024-03-02 08:30:00", "ZAR", "pos"),
    ("TXN006", "ACC103", 750.00, "credit", "Refund", "2024-03-02 14:00:00", "ZAR", "online"),
    ("TXN007", "ACC100", 120.00, "debit", "Uber", "2024-03-02 18:45:00", "ZAR", "online"),
    ("TXN008", "ACC102", 5000.00, "credit", "Salary Deposit", "2024-03-02 08:00:00", "ZAR", "eft"),
    ("TXN009", "ACC103", -50.00, "debit", "Error Entry", "2024-03-03 10:00:00", "ZAR", "pos"),
    ("TXN010", "ACC101", 200.00, "debit", "Grocery Store", "2024-03-03 11:30:00", "ZAR", "pos"),
    ("TXN003", "ACC100", 89.99, "debit", "Netflix", "2024-03-01 12:00:00", "ZAR", "online"),
    (None, "ACC100", 100.00, "debit", "Unknown", "2024-03-03 15:00:00", "ZAR", "pos"),
    ("TXN012", "ACC104", 0.00, "credit", "Zero Transfer", "2024-03-03 16:00:00", "ZAR", "eft"),
    ("TXN013", "ACC100", 350.00, "DEBIT", "  Pharmacy  ", "2024-03-04 09:00:00", "ZAR", "pos"),
    ("TXN014", "ACC102", 1800.00, "Credit", "Investment Return", "2024-03-04 10:00:00", "ZAR", "online"),
]

transactions_columns = [
    "transaction_id", "account_id", "amount", "transaction_type",
    "merchant", "timestamp", "currency", "channel"
]

df_transactions_raw = spark.createDataFrame(transactions_data, transactions_columns)
display(df_transactions_raw)

transaction_id,account_id,amount,transaction_type,merchant,timestamp,currency,channel
TXN001,ACC100,250.0,debit,SuperMart,2024-03-01 09:15:00,ZAR,pos
TXN002,ACC101,1500.0,credit,Salary Deposit,2024-03-01 08:00:00,ZAR,eft
TXN003,ACC100,89.99,debit,Netflix,2024-03-01 12:00:00,ZAR,online
TXN004,ACC102,3200.0,debit,Rent Payment,2024-03-01 07:00:00,ZAR,eft
TXN005,ACC101,45.5,debit,Coffee Shop,2024-03-02 08:30:00,ZAR,pos
TXN006,ACC103,750.0,credit,Refund,2024-03-02 14:00:00,ZAR,online
TXN007,ACC100,120.0,debit,Uber,2024-03-02 18:45:00,ZAR,online
TXN008,ACC102,5000.0,credit,Salary Deposit,2024-03-02 08:00:00,ZAR,eft
TXN009,ACC103,-50.0,debit,Error Entry,2024-03-03 10:00:00,ZAR,pos
TXN010,ACC101,200.0,debit,Grocery Store,2024-03-03 11:30:00,ZAR,pos


### Generate Simulated Customer Data

In [0]:
customers_data = [
    ("ACC100", "john smith", "JOHN.SMITH@EMAIL.COM", "0821234567", "10 Main Road", "Cape Town", "2020-01-15"),
    ("ACC101", "sarah jones", "sarah.jones@email.com", "0839876543", "25 Oak Avenue", "Johannesburg", "2019-06-20"),
    ("ACC102", "thabo mokoena", "THABO@WORK.CO.ZA", "0761112233", "8 Church Street", "Pretoria", "2021-03-10"),
    ("ACC103", "lisa van der berg", "lisa.vdb@gmail.com", "0844445566", "15 Beach Road", "Durban", "2022-08-01"),
    ("ACC104", "michael brown", "michael.b@email.com", None, "30 Hill Street", "Bloemfontein", "2023-01-05"),
    ("ACC100", "john smith", "john.smith@email.com", "0821234567", "10 Main Road", "Cape Town", "2020-01-15"),
]

customers_columns = [
    "customer_id", "name", "email", "phone", "address", "city", "account_open_date"
]

df_customers_raw = spark.createDataFrame(customers_data, customers_columns)
display(df_customers_raw)

customer_id,name,email,phone,address,city,account_open_date
ACC100,john smith,JOHN.SMITH@EMAIL.COM,0821234567,10 Main Road,Cape Town,2020-01-15
ACC101,sarah jones,sarah.jones@email.com,0839876543,25 Oak Avenue,Johannesburg,2019-06-20
ACC102,thabo mokoena,THABO@WORK.CO.ZA,0761112233,8 Church Street,Pretoria,2021-03-10
ACC103,lisa van der berg,lisa.vdb@gmail.com,0844445566,15 Beach Road,Durban,2022-08-01
ACC104,michael brown,michael.b@email.com,null,30 Hill Street,Bloemfontein,2023-01-05
ACC100,john smith,john.smith@email.com,0821234567,10 Main Road,Cape Town,2020-01-15


### Ingest into Bronze Delta Tables

In [0]:
batch_id = datetime.now().strftime("%Y%m%d%H%M%S")

df_transactions_bronze = df_transactions_raw \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_system", lit("core_banking")) \
    .withColumn("_batch_id", lit(batch_id))

df_transactions_bronze.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bronze.raw_transactions")

print(f"Bronze transactions ingested. Batch ID: {batch_id}")

Bronze transactions ingested. Batch ID: 20260625153639


In [0]:
df_customers_bronze = df_customers_raw \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_system", lit("core_banking")) \
    .withColumn("_batch_id", lit(batch_id))

df_customers_bronze.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bronze.raw_customers")

print(f"Bronze customers ingested. Batch ID: {batch_id}")

Bronze customers ingested. Batch ID: 20260625153639


In [0]:
bronze_txn_count = spark.table("bronze.raw_transactions").count()
bronze_cust_count = spark.table("bronze.raw_customers").count()
print(f"Bronze transactions: {bronze_txn_count} rows")
print(f"Bronze customers: {bronze_cust_count} rows")

Bronze transactions: 30 rows
Bronze customers: 12 rows


## Step 3: Silver Layer - Clean and Validate

### Data Quality Checks

In [0]:
df_bronze_txn = spark.table("bronze.raw_transactions")

null_ids = df_bronze_txn.filter(col("transaction_id").isNull())
negative_amounts = df_bronze_txn.filter(col("amount") < 0)
zero_credits = df_bronze_txn.filter(
    (col("transaction_type").ilike("credit")) & (col("amount") == 0)
)

window_dedup = Window.partitionBy("transaction_id").orderBy(col("_ingested_at").desc())
df_with_row_num = df_bronze_txn.filter(col("transaction_id").isNotNull()) \
    .withColumn("_row_num", row_number().over(window_dedup))
duplicates = df_with_row_num.filter(col("_row_num") > 1).drop("_row_num")

df_quarantine = null_ids.withColumn("_rejection_reason", lit("null_transaction_id")) \
    .unionByName(
        negative_amounts.withColumn("_rejection_reason", lit("negative_amount")),
        allowMissingColumns=True
    ) \
    .unionByName(
        zero_credits.withColumn("_rejection_reason", lit("zero_credit_amount")),
        allowMissingColumns=True
    )

df_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.quarantine_transactions")

total_records = df_bronze_txn.count()
quarantined = df_quarantine.count()
passed = total_records - quarantined

print("=== DATA QUALITY REPORT ===")
print(f"Total records:      {total_records}")
print(f"Passed:             {passed}")
print(f"Quarantined:        {quarantined}")
print(f"Pass rate:          {(passed/total_records)*100:.1f}%")

=== DATA QUALITY REPORT ===
Total records:      30
Passed:             24
Quarantined:        6
Pass rate:          80.0%


### Transform and Load Silver Transaction

In [0]:
df_clean_txn = df_with_row_num.filter(col("_row_num") == 1).drop("_row_num")

df_clean_txn = df_clean_txn \
    .filter(col("transaction_id").isNotNull()) \
    .filter(col("amount") > 0) \
    .withColumn("transaction_type", upper(trim(col("transaction_type")))) \
    .withColumn("merchant", trim(col("merchant"))) \
    .withColumn("amount", col("amount").cast("decimal(12,2)")) \
    .withColumn("transaction_timestamp", col("timestamp").cast("timestamp")) \
    .withColumn("transaction_date", to_date(col("timestamp"))) \
    .withColumn("processed_at", current_timestamp()) \
    .select(
        "transaction_id", "account_id", "amount", "transaction_type",
        "merchant", "transaction_timestamp", "transaction_date",
        "currency", "channel", "processed_at"
    )

df_clean_txn.createOrReplaceTempView("staged_transactions")

spark.sql("""
    CREATE TABLE IF NOT EXISTS silver.transactions (
        transaction_id STRING,
        account_id STRING,
        amount DECIMAL(12,2),
        transaction_type STRING,
        merchant STRING,
        transaction_timestamp TIMESTAMP,
        transaction_date DATE,
        currency STRING,
        channel STRING,
        processed_at TIMESTAMP
    ) USING DELTA
""")

spark.sql("""
    MERGE INTO silver.transactions AS target
    USING staged_transactions AS source
    ON target.transaction_id = source.transaction_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

silver_txn_count = spark.table("silver.transactions").count()
print(f"Silver transactions: {silver_txn_count} rows (clean, deduplicated)")

Silver transactions: 11 rows (clean, deduplicated)


### Transform and Load Silver Customers

In [0]:
df_bronze_cust = spark.table("bronze.raw_customers")

window_cust_dedup = Window.partitionBy("customer_id").orderBy(col("_ingested_at").desc())
df_clean_cust = df_bronze_cust \
    .withColumn("_row_num", row_number().over(window_cust_dedup)) \
    .filter(col("_row_num") == 1) \
    .drop("_row_num") \
    .withColumn("name", initcap(trim(col("name")))) \
    .withColumn("email", lower(trim(col("email")))) \
    .withColumn("phone", trim(col("phone"))) \
    .withColumn("address", trim(col("address"))) \
    .withColumn("city", initcap(trim(col("city")))) \
    .withColumn("account_open_date", col("account_open_date").cast("date")) \
    .withColumn("processed_at", current_timestamp()) \
    .select(
        "customer_id", "name", "email", "phone",
        "address", "city", "account_open_date", "processed_at"
    )

df_clean_cust.createOrReplaceTempView("staged_customers")

spark.sql("""
    CREATE TABLE IF NOT EXISTS silver.customers (
        customer_id STRING,
        name STRING,
        email STRING,
        phone STRING,
        address STRING,
        city STRING,
        account_open_date DATE,
        processed_at TIMESTAMP
    ) USING DELTA
""")

spark.sql("""
    MERGE INTO silver.customers AS target
    USING staged_customers AS source
    ON target.customer_id = source.customer_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

silver_cust_count = spark.table("silver.customers").count()
print(f"Silver customers: {silver_cust_count} rows (deduplicated, standardized)")
display(spark.table("silver.customers"))

Silver customers: 5 rows (deduplicated, standardized)


customer_id,name,email,phone,address,city,account_open_date,processed_at
ACC100,John Smith,john.smith@email.com,0821234567,10 Main Road,Cape Town,2020-01-15,2026-06-25T15:43:12.947Z
ACC101,Sarah Jones,sarah.jones@email.com,0839876543,25 Oak Avenue,Johannesburg,2019-06-20,2026-06-25T15:43:12.947Z
ACC102,Thabo Mokoena,thabo@work.co.za,0761112233,8 Church Street,Pretoria,2021-03-10,2026-06-25T15:43:12.947Z
ACC103,Lisa Van Der Berg,lisa.vdb@gmail.com,0844445566,15 Beach Road,Durban,2022-08-01,2026-06-25T15:43:12.947Z
ACC104,Michael Brown,michael.b@email.com,null,30 Hill Street,Bloemfontein,2023-01-05,2026-06-25T15:43:12.947Z


## Step 4: Gold Layer - Business Aggregations

### Daily Transaction Summary (Fact Table)

In [0]:
df_silver_txn = spark.table("silver.transactions")

df_daily_summary = df_silver_txn \
    .groupBy("transaction_date") \
    .agg(
        count("*").alias("total_transactions"),
        spark_sum("amount").alias("total_amount"),
        avg("amount").alias("avg_amount"),
        spark_sum(when(col("transaction_type") == "DEBIT", 1).otherwise(0)).alias("debit_count"),
        spark_sum(when(col("transaction_type") == "CREDIT", 1).otherwise(0)).alias("credit_count"),
        spark_sum(when(col("transaction_type") == "DEBIT", col("amount")).otherwise(0)).alias("total_debits"),
        spark_sum(when(col("transaction_type") == "CREDIT", col("amount")).otherwise(0)).alias("total_credits")
    ) \
    .orderBy("transaction_date")

df_daily_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fact_daily_transactions")

print("Gold: Daily transaction summary created")
display(spark.table("gold.fact_daily_transactions"))

Gold: Daily transaction summary created


transaction_date,total_transactions,total_amount,avg_amount,debit_count,credit_count,total_debits,total_credits
2024-03-01,4,5039.99,1259.997500,3,1,3539.99,1500.00
2024-03-02,4,5915.50,1478.875000,2,2,165.50,5750.00
2024-03-03,1,200.00,200.000000,1,0,200.00,0.00
2024-03-04,2,2150.00,1075.000000,1,1,350.00,1800.00


### Customer Dimension Table

In [0]:
df_silver_cust = spark.table("silver.customers")

window_sk = Window.orderBy("customer_id")
df_dim_customer = df_silver_cust \
    .withColumn("customer_sk", row_number().over(window_sk)) \
    .select("customer_sk", "customer_id", "name", "email", "city", "account_open_date")

df_dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_customer")

print("Gold: Customer dimension created")
display(spark.table("gold.dim_customer"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold: Customer dimension created


customer_sk,customer_id,name,email,city,account_open_date
1,ACC100,John Smith,john.smith@email.com,Cape Town,2020-01-15
2,ACC101,Sarah Jones,sarah.jones@email.com,Johannesburg,2019-06-20
3,ACC102,Thabo Mokoena,thabo@work.co.za,Pretoria,2021-03-10
4,ACC103,Lisa Van Der Berg,lisa.vdb@gmail.com,Durban,2022-08-01
5,ACC104,Michael Brown,michael.b@email.com,Bloemfontein,2023-01-05


### Customer Transaction Fact Table (Star Schema)

In [0]:
df_customer_facts = df_silver_txn \
    .groupBy("account_id") \
    .agg(
        count("*").alias("transaction_count"),
        spark_sum("amount").alias("total_spend"),
        avg("amount").alias("avg_transaction_value"),
        spark_sum(when(col("transaction_type") == "DEBIT", col("amount")).otherwise(0)).alias("total_debits"),
        spark_sum(when(col("transaction_type") == "CREDIT", col("amount")).otherwise(0)).alias("total_credits")
    )

df_dim = spark.table("gold.dim_customer")

df_fact_customer_txn = df_customer_facts \
    .join(df_dim, df_customer_facts["account_id"] == df_dim["customer_id"], "left") \
    .select(
        df_dim["customer_sk"],
        df_customer_facts["account_id"],
        "transaction_count",
        "total_spend",
        "avg_transaction_value",
        "total_debits",
        "total_credits"
    )

df_fact_customer_txn.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fact_customer_transactions")

print("Gold: Customer transaction facts created (star schema)")
display(spark.table("gold.fact_customer_transactions"))

Gold: Customer transaction facts created (star schema)


customer_sk,account_id,transaction_count,total_spend,avg_transaction_value,total_debits,total_credits
4,ACC103,1,750.00,750.000000,0.00,750.00
1,ACC100,4,809.99,202.497500,809.99,0.00
2,ACC101,3,1745.50,581.833333,245.50,1500.00
3,ACC102,3,10000.00,3333.333333,3200.00,6800.00


## Step 5: Data Quality Gates and Time Travel

### Pipeline Quality Assertions

In [0]:
checks = []

bronze_count = spark.table("bronze.raw_transactions").count()
checks.append(("Bronze has data", bronze_count > 0, f"{bronze_count} rows"))

silver_nulls = spark.table("silver.transactions").filter(col("transaction_id").isNull()).count()
checks.append(("Silver has no null IDs", silver_nulls == 0, f"{silver_nulls} nulls"))

silver_total = spark.table("silver.transactions").agg(spark_sum("amount")).collect()[0][0]
gold_total = spark.table("gold.fact_daily_transactions").agg(spark_sum("total_amount")).collect()[0][0]
amounts_match = abs(float(silver_total) - float(gold_total)) < 0.01
checks.append(("Gold totals match Silver", amounts_match, f"Silver={silver_total}, Gold={gold_total}"))

print("=== PIPELINE HEALTH REPORT ===")
print("-" * 50)
for check_name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {check_name} ({detail})")
print("-" * 50)
all_passed = all(passed for _, passed, _ in checks)
print(f"Overall: {'ALL CHECKS PASSED' if all_passed else 'SOME CHECKS FAILED'}")

=== PIPELINE HEALTH REPORT ===
--------------------------------------------------
[PASS] Bronze has data (30 rows)
[PASS] Silver has no null IDs (0 nulls)
[PASS] Gold totals match Silver (Silver=13305.49, Gold=13305.49)
--------------------------------------------------
Overall: ALL CHECKS PASSED


### Delta Lake Time Travel

In [0]:
# View table history
display(spark.sql("DESCRIBE HISTORY silver.transactions"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-06-25T15:42:29.000Z,77899660694852,lindokuhle.nyoka03@gmail.com,MERGE,"Map(predicate -> [""(transaction_id#13026 = transaction_id#12966)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1556675641186801),d28dc621-a7c0-47f9-a4d1-b2eb416b9d08,0625-153558-r98dlo63-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3450, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 4725, materializeSourceTimeMs -> 961, numTargetRowsInserted -> 11, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2138, numTargetRowsUpdated -> 0, numOutputRows -> 11, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 11, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1480)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
0,2026-06-25T15:42:23.000Z,77899660694852,lindokuhle.nyoka03@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-96224cb1-7f01-4ab6-8a30-ed0ed4597beb"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-c6665ace-963d-41f4-8e80-4464e42828e8""}, statsOnLoad -> false)",null,List(1556675641186801),e11a0a00-1b45-4968-a0ac-dbb76f9ebd16,0625-153558-r98dlo63-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13


In [0]:
# Query the first version of the table (before any MERGE operations)
try:
    df_version_0 = spark.sql("SELECT * FROM silver.transactions VERSION AS OF 0")
    print(f"Version 0 had {df_version_0.count()} rows")
    display(df_version_0)
except Exception as e:
    print(f"Version 0 query: {e}")

# Current version
df_current = spark.table("silver.transactions")
print(f"Current version has {df_current.count()} rows")

Version 0 had 0 rows


transaction_id,account_id,amount,transaction_type,merchant,transaction_timestamp,transaction_date,currency,channel,processed_at


Current version has 11 rows


### Pipeline Complete

You have built a full medallion architecture pipeline:
- Bronze: Raw ingestion with metadata (append-only)
- Silver: Cleaned, validated, deduplicated (MERGE INTO)
- Gold: Business aggregations and star schema
- Quality gates and time travel audit trail